# MiniGPT tiếng Việt: tự huấn luyện GPT từ đầu
Cùng kiến trúc GPT-2, ~34 triệu tham số, học trên Wikipedia tiếng Việt rồi được dạy trò chuyện.

**Trước tiên:** *Runtime → Change runtime type → T4 GPU → Save*. Chạy lần lượt từng ô (▶). Tổng thời gian khoảng 3 giờ, chia được thành nhiều lần (tự tiếp tục).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Lấy mã nguồn
Kéo file `mini-gpt.zip` vào khung **Files** bên trái rồi chạy ô dưới (hoặc để ô tự lấy từ GitHub nếu repo công khai).

In [ ]:
import os
%cd /content
if os.path.exists('mini-gpt.zip'):
    !unzip -oq mini-gpt.zip
elif not os.path.exists('mini-gpt'):
    !git clone -q --depth 1 -b ccr-b6e744b3-y0noaa https://github.com/thanhdangkhoa16122008-hash/TDK tdk && mv tdk/mini-gpt mini-gpt
assert os.path.exists('mini-gpt/train.py'), 'Chưa có mã nguồn: kéo mini-gpt.zip vào khung Files rồi chạy lại.'
%cd /content/mini-gpt
!pip install -q datasets gradio
!python tests/test_minigpt.py

## 2. Lưu vào Google Drive
Dữ liệu và mô hình được lưu vào Drive. Colab có ngắt thì chạy lại các ô, việc huấn luyện **tự tiếp tục** từ chỗ dừng.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
D = '/content/drive/MyDrive/minigpt'
os.makedirs(D, exist_ok=True)
print('Lưu tại', D)

## 3. Chuẩn bị dữ liệu (~10–20 phút, chỉ làm 1 lần)
Tải 150.000 bài Wikipedia tiếng Việt, học bộ tách từ 16.000 token, mã hóa toàn bộ. Muốn thêm văn bản của bạn (truyện, kịch bản…): tải các file .txt lên thư mục `mini-gpt/corpus/` rồi thêm `--input corpus`.

In [ ]:
if not os.path.exists(f'{D}/data-vi/train.bin'):
    !python prepare.py --wiki 150000 --vocab 16000 --out $D/data-vi --workers 2
else:
    print('Đã có dữ liệu, bỏ qua.')

## 4. Tiền huấn luyện (~2,5 giờ, chạy lại ô để học tiếp)
Theo dõi `loss val`: càng thấp mô hình viết càng tự nhiên. Mỗi lần đánh giá, mô hình viết thử một câu (✍) để bạn thấy nó tiến bộ.

In [ ]:
!python train.py --data $D/data-vi --out $D/out --preset small --max-iters 12000 --time-limit 170 --eval-every 500

In [ ]:
# Viết tiếp văn bản bằng mô hình vừa học
!python chat.py --ckpt $D/out --prompt "Hà Nội là" --tokens 120

## 5. Dạy trò chuyện (~5 phút)
Dùng 1.139 hội thoại tiếng Việt trong `data/sft_vi.jsonl`. Thêm hội thoại của bạn vào file này (cùng định dạng) để Mây nói chuyện theo ý bạn.

In [ ]:
!python sft.py --base $D/out --data data/sft_vi.jsonl --out $D/out-chat --epochs 3

## 6. Trò chuyện
Bấm link `…gradio.live` hiện ra (mở được cả trên điện thoại).

In [ ]:
!python chat.py --ckpt $D/out-chat --web --share